In [ ]:
from rcsb.data import QueryBuilder as QB, Query
from rcsb import unwrap_query

In [ ]:
query = (QB().entry(entry_id="$id") # NOTE: kwarg matches entry(entry_id: String!)
    .polymer_entities
        .rcsb_id
        .entity_poly
            .pdbx_seq_one_letter_code_can
            .end
        .rcsb_target_cofactors
            .binding_assay_value
            .binding_assay_value_type
            .cofactor_SMILES
            .end # move back up to polymer_entities
        .uniprots
            .rcsb_id
            .end # move back up to polymer_entities
        .rcsb_polymer_entity_align
            .aligned_regions
                .entity_beg_seq_id
                .ref_beg_seq_id
                .end # move back up to rcsb_polymer_entity_align
            .end # move back up to polymer_entities
        .polymer_entity_instances
            .rcsb_polymer_instance_feature
                .name
                .feature_positions
                    .beg_comp_id
                    .beg_seq_id
                .end # optional; move back up to rcsb_polymer_instance_feature
            .end # optional; move back up to polymer_entity_instances
        .end # optional; move back to polymer_entities
    .end # optional; move back up to entry (crucial for .process(), can be checked with ._name())
    )

In [ ]:
print(query.render())

In [ ]:
query.submit(id="1b38") # errors if any keyword other than id is used because '$id' was used in query

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import os

rendered_query = query.render()
target_ids = ["1b38", "6mdr", "5dwy"]

results = []
MAX_WORKERS = os.cpu_count()
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {
        executor.submit(
            Query.execute,
            rendered_query, # must be rendered outside, as render() modifies self
            **{"id": pdb_id.lower()}
        ): pdb_id.lower() for pdb_id in target_ids
    }

for future in as_completed(futures):
    pdb_id = futures[future]
    try:
        data = future.result()
        results.append(data)
    except Exception as e:
        print(f"{pdb_id} generated an exception: {e}")

print(results)

In [ ]:
query_many = (QB().entries(entry_ids="$ids") # NOTE: kwarg matches entry(entry_id: String!)
    .polymer_entities
        .rcsb_id
        .entity_poly
            .pdbx_seq_one_letter_code_can
            .end
        .rcsb_target_cofactors
            .binding_assay_value
            .binding_assay_value_type
            .cofactor_SMILES
            .end # move back up to polymer_entities
        .uniprots
            .rcsb_id
            .end # move back up to polymer_entities
        .rcsb_polymer_entity_align
            .aligned_regions
                .entity_beg_seq_id
                .ref_beg_seq_id
                .end # move back up to rcsb_polymer_entity_align
            .end # move back up to polymer_entities
        .polymer_entity_instances
            .rcsb_polymer_instance_feature
                .name
                .feature_positions
                    .beg_comp_id
                    .beg_seq_id
                    .end
                .end
            .end 
        .end
    .nonpolymer_entities
        .nonpolymer_comp
            .chem_comp.id.end
            .end
        .nonpolymer_entity_instances
            .rcsb_nonpolymer_instance_validation_score
                .is_subject_of_investigation
                .end
            .end
        .end
    .end
)

In [ ]:
query_many.render()

In [ ]:
pdb_ids = [
    "1A02", "1A8W", "1AFW", "1AIE", "1AOI", "1ATW", "1B31", "1B8G", "1BL8", "1BNA",
    "1C8C", "1CRN", "1D66", "1EMA", "1F9J", "1FAT", "1G01", "1GFL", "1GZX", "1HHO",
    "1J4N", "1K4C", "1L2Y", "1MBO", "1NA0", "1O17", "1PDB", "1R62", "1SBT", "1TUP",
    "1UBQ", "1XWW", "1Y64", "1Z7I", "2A7D", "2BIW", "2CHB", "2DHB", "2HVP", "2L7M",
    "2M6B", "2NN6", "2OAU", "2P1L", "2POR", "2RH1", "2VAA", "2W9E", "2XHE", "2YGD",
    "3A0G", "3B75", "3C1X", "3D29", "3E7E", "3F82", "3G6D", "3H9R", "3I40", "3J3Y",
    "3K7U", "3L2J", "3M7G", "3N8L", "3O9S", "3P4J", "3Q6M", "3R7X", "3S8Y", "3T9Z",
    "4A0C", "4B1D", "4C2E", "4D3F", "4E4G", "4F5H", "4G6I", "4H7J", "4I8K", "4J9L",
    "4K0M", "4L1N", "4M2O", "4N3P", "4O4Q", "4P5R", "4Q6S", "4R7T", "4S8U", "4T9V",
    "5A0W", "5B1X", "5C2Y", "5D3Z", "5E4A", "5F5B", "5G6C", "5H7D", "5I8E", "6LU7"
]


def process_single_entry(entry, pdb_id: str, ligand: str = None, filter_affinity_nulls: bool = True):
    if entry is None:
        print(f"Query not found for {pdb_id}. Setting value to None")
        return None

    nonpolymer_entities = entry.get("nonpolymer_entities")

    subject_of_investigation = None
    if ligand is not None:
        subject_of_investigation = ligand
    else:
        for entity in nonpolymer_entities or tuple():
            is_subject_of_investigation = unwrap_query(entity,
                                                  path=[
                                                      "nonpolymer_entity_instances",
                                                      "rcsb_nonpolymer_instance_validation_score",
                                                      "is_subject_of_investigation"
                                                  ])
            if is_subject_of_investigation == 'Y':
                subject_of_investigation = entity.get("nonpolymer_comp").get("chem_comp").get("id")


    polymer_entity = entry.get("polymer_entities")[0]
    canonical_seq = unwrap_query(polymer_entity, ["entity_poly", "pdbx_seq_one_letter_code_can"])

    uniprot_offset = None
    uniprot_idx = None
    uniprot_id = None
    try:
        uniprot_id = unwrap_query(polymer_entity, ["uniprots", "rcsb_id"])
        aligned_regions = unwrap_query(polymer_entity, ["rcsb_polymer_entity_align", "aligned_regions"])
        uniprot_offset = unwrap_query(aligned_regions, ["entity_beg_seq_id"]) - 1
        uniprot_idx = unwrap_query(aligned_regions, ["ref_beg_seq_id"])
    except Exception as e:
        print(f"Warning: {pdb_id} uniprot detection failed with {type(e)} {e}")

    ## cofactor smiles
    cofactors = polymer_entity.get("rcsb_target_cofactors")
    if filter_affinity_nulls and cofactors is not None:
        cofactors = [cofactor for cofactor in cofactors if cofactor.get("binding_assay_value") is not None] 

    smiles = list()
    affinity = list()
    affinity_type = list()
    for cofactor in cofactors or tuple():
        smiles.append(cofactor["cofactor_SMILES"])
        affinity.append(cofactor["binding_assay_value"])
        affinity_type.append(cofactor["binding_assay_value_type"])


    return dict(
        pdb_id=pdb_id,
        seq=canonical_seq,
        uniprot_offset=uniprot_offset,
        uniprot_index=uniprot_idx,
        uniprot_id=uniprot_id,
        smiles=tuple(smiles),
        affinity=tuple(affinity),
        affinity_type=tuple(affinity_type),
        ligand=subject_of_investigation,
    )

In [ ]:
query_many.process(pdb_ids, process_single_entry, iter_kwargs={"pdb_id": pdb_ids})

In [ ]:
process_single_entry(query.submit(id="1b38").get("entry"), "1b38")